In [103]:
import pandas as pd 
import numpy as np
import psycopg2

In [11]:
!pip install  psycopg2-binary

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 4.8 MB/s eta 0:00:01
   -------------------------- ------------- 1.8/2.8 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 5.4 MB/s  0:00:00


In [104]:
conn = psycopg2.connect(
    host="localhost",
    database="dammy_project",
    user="postgres",
    password="7523Abdulla&",
    port=5432
)

print("Connected to PostgreSQL successfully!")

Connected to PostgreSQL successfully!


In [17]:
def connect_db():
  return psycopg2.connect(
    host="localhost",
    database="dammy_project",
    user="postgres",
    password="7523Abdulla&",
    port=5432
)

In [30]:
db = connect_db()
from psycopg2.extras import RealDictCursor

cursor = db.cursor(cursor_factory=RealDictCursor)


In [29]:
cursor

<cursor object at 0x00000244F32D1E50; closed: 0>

In [31]:
"select count(*) as Total_suplier from suppliers"

'select count(*) as Total_suplier from suppliers'

In [32]:
cursor.execute("select count(*) as Total_suplier from suppliers")

In [33]:
row = cursor.fetchone()

In [34]:
row

RealDictRow([('total_suplier', 50)])

In [36]:
list(row.values())[0]

50

In [47]:
queries = {
 "Total suplier":   "select count(*) as Total_suplier from suppliers",
    
  "Total products":  "select count(*) as Total_product from products",
    
  "Total catagori dealing":  "select count(distinct category) as total_category from products",
    
   "Total sales values  made in last 3 months": """select round(sum(abs(se.change_quantity)*p.price),2) as total_sales_values_last_3_moths
from stock_entries as se
join products p
on p.product_id = se.product_id
where se.change_type = 'OUT'
and se.entry_date>= 
        (
          SELECT MAX(entry_date) - INTERVAL '3 months'
           FROM stock_entries

		)""",
    
   "Total restock values  made in last 3 months": """ 
select round(sum(abs(se.change_quantity)*p.price),2) as total_restock_values_last_3_moths
from stock_entries as se
join products p
on p.product_id = se.product_id
where se.change_type = 'IN'
and se.entry_date>= 
        (
          SELECT MAX(entry_date) - INTERVAL '3 months'
           FROM stock_entries

		)""",

   "Below reorder no pending records" : """ 
select count(*) from products as p where p.stock_quantity<p.reorder_level
and product_id not in
(
select distinct product_id from reorders where status = 'Pending'
)
"""}



In [50]:
result={}
for label , query in queries.items():
    cursor.execute(query)
    row = cursor.fetchone()
    result[label]=list(row.values())[0]
    
    

In [51]:
result

{'Total suplier': 50,
 'Total products': 50,
 'Total catagori dealing': 5,
 'Total sales values  made in last 3 months': Decimal('194240.00'),
 'Total restock values  made in last 3 months': Decimal('4829770.00'),
 'Below reorder no pending records': 0}

In [57]:
def get_basic_info(cursor):
    queries = {
 
    "Total suplier": "select count(*) as Total_suplier from suppliers",

    "Total products": "select count(*) as Total_product from products",

    "Total catagori dealing": 
        "select count(distinct category) as total_category from products",

    "Total sales values made in last 3 months": """
        select round(sum(abs(se.change_quantity)*p.price), 2) 
        as total_sales_values_last_3_months
        from stock_entries as se
        join products p on p.product_id = se.product_id
        where se.change_type = 'OUT'
        and se.entry_date >= (
            select max(entry_date) - interval '3 months'
            from stock_entries
        )
    """,

    "Total restock values made in last 3 months": """
        select round(sum(abs(se.change_quantity)*p.price), 2) 
        as total_restock_values_last_3_months
        from stock_entries as se
        join products p on p.product_id = se.product_id
        where se.change_type = 'IN'
        and se.entry_date >= (
            select max(entry_date) - interval '3 months'
            from stock_entries
        )
    """,

    "Below reorder no pending records": """
        select count(*)
        from products as p
        where p.stock_quantity < p.reorder_level
        and product_id not in (
            select distinct product_id
            from reorders
            where status = 'Pending'
        )
    """
         }

    result={}
    for label , query in queries.items():
       cursor.execute(query)
       row = cursor.fetchone()
        # row is a dicstionary , extrect a single value by getting first value in dict.value()
       result[label]=list(row.values())[0]
    
    
    return result

In [90]:
queries = {

    
"Supplier and there contact details" : "select  supplier_name , contact_name , email , phone from suppliers",

"Product with there suplliers and current work":
"""select p.product_name , s.supplier_name  , p.stock_quantity , p.reorder_level
from products as p
join suppliers  s on 
p.supplier_id = s.supplier_id
order by p.product_name asc""",


"product needing records": "select  product_name , stock_quantity , reorder_level  from products where stock_quantity<=reorder_level"

}
tables = {}
for lable , query in queries.items():
       cursor.execute(query)
       tables[lable]=cursor.fetchall()

In [91]:
tables

{'Supplier and there contact details': [RealDictRow([('supplier_name',
                'Alpha Traders'),
               ('contact_name', 'Rahul Sharma'),
               ('email', 'alpha1@gmail.com'),
               ('phone', '9876543201')]),
  RealDictRow([('supplier_name', 'Beta Supplies'),
               ('contact_name', 'Amit Verma'),
               ('email', 'beta2@gmail.com'),
               ('phone', '9876543202')]),
  RealDictRow([('supplier_name', 'Gamma Foods'),
               ('contact_name', 'Rakesh Patel'),
               ('email', 'gamma3@gmail.com'),
               ('phone', '9876543203')]),
  RealDictRow([('supplier_name', 'Omega Electronics'),
               ('contact_name', 'Suresh Kumar'),
               ('email', 'omega4@gmail.com'),
               ('phone', '9876543204')]),
  RealDictRow([('supplier_name', 'Delta Distributors'),
               ('contact_name', 'Ankit Jain'),
               ('email', 'delta5@gmail.com'),
               ('phone', '9876543205')]),
  Re

In [93]:
def get_additional_table(cursor):
    queries = {

    
       "Supplier and there contact details" : "select  supplier_name , contact_name , email , phone from suppliers",

      "Product with there suplliers and current work":
      """select p.product_name , s.supplier_name  , p.stock_quantity , p.reorder_level
      from products as p
      join suppliers  s on 
       p.supplier_id = s.supplier_id
       order by p.product_name asc""",


      "product needing records": "select  product_name , stock_quantity , reorder_level  from products where stock_quantity<=reorder_level"

            }
    tables = {}
    for lable , query in queries.items():
       cursor.execute(query)
       tables[lable]=cursor.fetchall()
    return tables

In [95]:
get_additional_table(cursor)

{'Supplier and there contact details': [RealDictRow([('supplier_name',
                'Alpha Traders'),
               ('contact_name', 'Rahul Sharma'),
               ('email', 'alpha1@gmail.com'),
               ('phone', '9876543201')]),
  RealDictRow([('supplier_name', 'Beta Supplies'),
               ('contact_name', 'Amit Verma'),
               ('email', 'beta2@gmail.com'),
               ('phone', '9876543202')]),
  RealDictRow([('supplier_name', 'Gamma Foods'),
               ('contact_name', 'Rakesh Patel'),
               ('email', 'gamma3@gmail.com'),
               ('phone', '9876543203')]),
  RealDictRow([('supplier_name', 'Omega Electronics'),
               ('contact_name', 'Suresh Kumar'),
               ('email', 'omega4@gmail.com'),
               ('phone', '9876543204')]),
  RealDictRow([('supplier_name', 'Delta Distributors'),
               ('contact_name', 'Ankit Jain'),
               ('email', 'delta5@gmail.com'),
               ('phone', '9876543205')]),
  Re

In [109]:
def add_new_manual_id(cursor , db , p_name,p_category, p_price ,p_stock, p_reorder, p_supplier):
    proc_call = " select add_new_product_manual_id(%s,%s,%s ,%s, %s,%s)"
    params = ( p_name,p_category, p_price ,p_stock, p_reorder, p_supplier)
    cursor.execute(proc_call , params)
    db.commit()

In [112]:
def  get_categories(cursor):
    cursor.execute("select distinct category from products order by category asc")
    rows = cursor.fetchall()
    return [row["category"] for row in rows]

In [115]:
get_categories(cursor)

['Electronics', 'Furniture', 'Grocery', 'Stationery', 'Utilities']

In [118]:
e

In [121]:
get_suppliers(cursor)

[RealDictRow([('supplier_id', 1), ('supplier_name', 'Alpha Traders')]),
 RealDictRow([('supplier_id', 2), ('supplier_name', 'Beta Supplies')]),
 RealDictRow([('supplier_id', 25), ('supplier_name', 'Budget Mart')]),
 RealDictRow([('supplier_id', 48), ('supplier_name', 'Business Link')]),
 RealDictRow([('supplier_id', 22), ('supplier_name', 'Central Goods')]),
 RealDictRow([('supplier_id', 16), ('supplier_name', 'City Mart')]),
 RealDictRow([('supplier_id', 12), ('supplier_name', 'Daily Essentials')]),
 RealDictRow([('supplier_id', 5), ('supplier_name', 'Delta Distributors')]),
 RealDictRow([('supplier_id', 21), ('supplier_name', 'East Hub')]),
 RealDictRow([('supplier_id', 26), ('supplier_name', 'Elite Traders')]),
 RealDictRow([('supplier_id', 8), ('supplier_name', 'Everest Goods')]),
 RealDictRow([('supplier_id', 30), ('supplier_name', 'Express Traders')]),
 RealDictRow([('supplier_id', 24), ('supplier_name', 'Fast Track Supply')]),
 RealDictRow([('supplier_id', 11), ('supplier_name',

In [120]:
db.rollback()
